In [3]:
import argparse
from dataclasses import asdict, dataclass, field
from pathlib import Path
import random
import sys
from typing import Any, Dict, List, Optional, Tuple

import gymnasium as gym
import numpy as np

import rclpy

import tb4_drl_navigation.envs  # noqa: F401
import torch
import torch.nn as nn
import time
import yaml
from transforms3d.euler import quat2euler, euler2quat

import rclpy
from rclpy.node import Node
from nav_msgs.msg import Odometry


ModuleNotFoundError: No module named 'tb4_drl_navigation'

In [ ]:
def get_path(path_type: str):
    if path_type == "line":
        start_pos = (0, 0, 0) # (x, y, yaw)
        goal_pos = (10, 0, 0)
        path = np.vstack([
                10 * np.linspace(0, 100, 5000),
                0 * np.ones(5000)
            ])
        is_loop=False
    return start_pos, goal_pos, path, is_loop


# TODO cite racecar
def get_closest_index(path: np.ndarray, x: float, y: float, start_ind: int, is_loop: bool):
    best_dist = np.inf
    
    ind_s = [i for i in range(start_ind)]
    ind_e = [start_ind + i for i in range(path.shape[1])]

    if not is_loop:
        indices = ind_e
    else:
        indices = [i for a in [ind_e, ind_s] for i in a]

    for i in indices:
        path_x = path[0, i]
        path_y = path[1, i]

        dist = np.sqrt((path_x - x)**2 + (path_y - y)**2)

        if dist < best_dist:
            best_dist = dist
        else:
            break
    return i

def get_horizon_xy(path: np.ndarray, current_index: float, horizon: int):
    max_index = path.shape[1] - 1
    
    if current_index + horizon > max_index:
        index =  max_index
    else:
        index = current_index + horizon

    return np.array([path[0, index], path[1, index]])

def get_robot_xy(env):
    # Developed based on turtlebot4 _get_odom code
    # Get current pose
    pose_stamped = env.sensors.get_latest_pose_stamped()
    agent_pose = pose_stamped.pose

    # Extract positions
    agent_x = agent_pose.position.x
    agent_y = agent_pose.position.y

    # Extract current orientation
    q = [
        agent_pose.orientation.w,
        agent_pose.orientation.x,
        agent_pose.orientation.y,
        agent_pose.orientation.z
    ]
    _, _, yaw = quat2euler(q, 'sxyz')
    return np.array([agent_x, agent_y]), yaw

class VelocityListener(Node):
    def __init__(self):
        super().__init__('velocity_listener')

        self.linear_velocity = 0.0

        self.sub = self.create_subscription(
            Odometry,
            '/odom',   # change if needed
            self.callback,
            10
        )

    def callback(self, msg):
        self.linear_velocity = msg.twist.twist.linear.x

In [1]:
def run_controller(pid_config_file: str, world: str):
    with open(pid_config_file, 'r') as file:
        pid_config = yaml.safe_load(file)

    print("loaded_config")

    rclpy.init()

    env=gym.make('Turtlebot4Env-v0')
    node = VelocityListener()

    start_pos, goal_pos, path, is_loop = get_path(pid_config["path"])

    kp = pid_config["kp"]
    ki = pid_config["ki"]
    kd = pid_config["kd"]
    speed = pid_config["speed"]
    horizon = pid_config["horizon"]

    state = env.reset(options={"start_pos": start_pos, "goal_pos": goal_pos})
    print("env made")

        
    xy_err = [0]
    prev_xy_err = 0

    speed_err = [0]
    prev_speed_err=0
    
    rewards = []
    observation, reward, terminated, truncated, info = env.step(np.ndarray([1, 0]))

    robot_pos = []
    

    curr_index = 0
    while (not terminated):
        # print(i)
        rclpy.spin_once(node, timeout_sec=0)
        
        xy, yaw = get_robot_xy(env)

        curr_index = get_closest_index(path, xy[0], xy[1], curr_index, is_loop)

        horizon_xy = get_horizon_xy(path, curr_index, horizon)

        (dx, dy) = (horizon_xy - xy)

        diff = np.arctan2(dy, dx)
        
        curr_xy_err = diff - yaw

        # make sure angle is in the possible range of angles
        if curr_xy_err > np.pi:
            curr_xy_err = curr_xy_err - 2*np.pi
        elif curr_xy_err < -np.pi:
            curr_xy_err = curr_xy_err + 2*np.pi

        prev_xy_err = xy_err[-1]
        xy_err.append(curr_xy_err)
        
        prev_speed_err = speed_err[-1]
        curr_speed_err = speed - node.linear_velocity
        speed_err.append(curr_speed_err)

        steer = kp[0] * curr_xy_err + ki[0] * sum(xy_err) * 0.05 + kd[0] * (curr_xy_err - prev_xy_err) / 0.05
        thrust = kp[1] * (curr_speed_err) + ki[1] * sum(speed_err) * 0.05 + kd[0] * (curr_speed_err-prev_speed_err) /0.05
 
        observation, reward, terminated, truncated, info = env.step(np.ndarray([steer, thrust]))

        rewards.append(reward)
        robot_pos.append(xy)



    return robot_pos, xy_err, speed_err, rewards